In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Y2O3 — neutron powder, constant wavelength, anisotropic β-tensor ADPs

Cross-engine check of the **dimensionless β-tensor** anisotropic ADP
convention (`adp_type='beta'`) and the cylindrical Debye–Scherrer
sample-absorption correction on a powder pattern.

The **only** parameter that does not match is the Bérar–Baldinozzi
axial-divergence asymmetry: cryspy and FullProf implement it with
different conventions (an overall sign and a coefficient inside the
`F_b` term differ), so the `asym_beba_*` coefficients do not transfer
one-to-one between the two programs. Freeing them in the fit absorbs
the difference; the structural results are unaffected. See development
issue 166 for the detailed comparison.

In [2]:
import easydiffraction as edi
from easydiffraction import ExperimentFactory
from easydiffraction import StructureFactory
from easydiffraction.analysis import verification as verify

## Build the project

In [3]:
project = edi.Project()

## Define the structure

Occupancies are the crystallographic site fractions (all fully
occupied). FullProf's `.pcr` lists the multiplicity-weighted values
(0.5, 0.16667, 1.0 for the 24d, 8b and 48e sites); cryspy derives the
site multiplicity from the symmetry, so the fractions are 1.0 here.

In [4]:
structure = StructureFactory.from_scratch(name='y2o3')

structure.space_group.name_h_m = 'I a -3'  # FullProf Space group symbol

structure.cell.length_a = 10.605744  # FullProf a

structure.atom_sites.create(
    id='Y1',  # FullProf Atom
    type_symbol='Y',  # FullProf Typ
    fract_x=-0.03236,  # FullProf X
    fract_y=0.0,  # FullProf Y
    fract_z=0.25,  # FullProf Z
    occupancy=1.0,  # FullProf Occ 0.50000 (24d site)
    adp_type='beta',  # FullProf N_t = 2 (anisotropic β)
)
structure.atom_sites.create(
    id='Y2',  # FullProf Atom
    type_symbol='Y',  # FullProf Typ
    fract_x=0.25,  # FullProf X
    fract_y=0.25,  # FullProf Y
    fract_z=0.25,  # FullProf Z
    occupancy=1.0,  # FullProf Occ 0.16667 (8b site)
    adp_type='beta',  # FullProf N_t = 2 (anisotropic β)
)
structure.atom_sites.create(
    id='O1',  # FullProf Atom
    type_symbol='O',  # FullProf Typ
    fract_x=0.39072,  # FullProf X
    fract_y=0.15204,  # FullProf Y
    fract_z=0.38030,  # FullProf Z
    occupancy=1.0,  # FullProf Occ 1.00000 (48e site)
    adp_type='beta',  # FullProf N_t = 2 (anisotropic β)
)

# β tensor components (FullProf Betas), dimensionless.
structure.atom_site_aniso['Y1'].adp_11 = 0.00303  # FullProf beta11
structure.atom_site_aniso['Y1'].adp_22 = 0.00272  # FullProf beta22
structure.atom_site_aniso['Y1'].adp_33 = 0.00295  # FullProf beta33
structure.atom_site_aniso['Y1'].adp_23 = -0.00025  # FullProf beta23

structure.atom_site_aniso['Y2'].adp_11 = 0.00304  # FullProf beta11
structure.atom_site_aniso['Y2'].adp_12 = -0.00013  # FullProf beta12

structure.atom_site_aniso['O1'].adp_11 = 0.00299  # FullProf beta11
structure.atom_site_aniso['O1'].adp_22 = 0.00310  # FullProf beta22
structure.atom_site_aniso['O1'].adp_33 = 0.00273  # FullProf beta33
structure.atom_site_aniso['O1'].adp_12 = -0.00007  # FullProf beta12
structure.atom_site_aniso['O1'].adp_13 = -0.00020  # FullProf beta13
structure.atom_site_aniso['O1'].adp_23 = -0.00001  # FullProf beta23

project.structures.add(structure)

## Load the FullProf reference

In [5]:
FULLPROF_PROJECT_DIR = 'pd-neut-cwl_pv-beta_y2o3'
FULLPROF_PRF_FILE = 'y2o3.prf'
FULLPROF_BAC_FILE = 'y2o3.bac'
FULLPROF_SUM_FILE = 'y2o3.sum'
FULLPROF_LABEL = verify.fullprof_label(FULLPROF_PROJECT_DIR, FULLPROF_SUM_FILE)
FULLPROF_ZERO = -0.01625  # FullProf Zero
FULLPROF_SCALE = 1.0602  # FullProf Scale
FULLPROF_WAVELENGTH = 1.54822  # FullProf Lambda
FULLPROF_U = 0.036631  # FullProf U
FULLPROF_V = -0.068345  # FullProf V
FULLPROF_W = 0.131426  # FullProf W
FULLPROF_ASY_1 = 0.17694  # FullProf Asy1
FULLPROF_ASY_2 = 0.03411  # FullProf Asy2
FULLPROF_MU_R = 1.5  # FullProf muR

x, calc_fullprof = verify.load_fullprof_calc_profile(
    FULLPROF_PROJECT_DIR,
    FULLPROF_PRF_FILE,
    FULLPROF_BAC_FILE,
    FULLPROF_ZERO,
)

## Create the experiment

In [6]:
experiment = ExperimentFactory.from_scratch(
    name='y2o3',
    sample_form='powder',
    beam_mode='constant wavelength',
    radiation_probe='neutron',
    scattering_type='bragg',
)
verify.set_reference_as_measured(experiment, x, calc_fullprof)

experiment.linked_structures.create(structure_id='y2o3', scale=FULLPROF_SCALE)

experiment.instrument.setup_wavelength = FULLPROF_WAVELENGTH
experiment.instrument.calib_twotheta_offset = FULLPROF_ZERO

experiment.peak.type = 'pseudo-voigt + berar-baldinozzi asymmetry'
experiment.peak.broad_gauss_u = FULLPROF_U
experiment.peak.broad_gauss_v = FULLPROF_V
experiment.peak.broad_gauss_w = FULLPROF_W
experiment.peak.asym_beba_a0 = FULLPROF_ASY_1
experiment.peak.asym_beba_b0 = FULLPROF_ASY_2

experiment.absorption.type = 'cylinder-hewat'
experiment.absorption.mu_r = FULLPROF_MU_R

# FullProf excluded the 0–12° and 137.5–180° regions (.pcr).
experiment.excluded_regions.create(id='1', start=0.0, end=12.0)
experiment.excluded_regions.create(id='2', start=137.5, end=180.0)

project.experiments.add(experiment)

⚠️ Switching peak profile type adds these settings with defaults:                                                                 
   • asym_beba_a0=0.0                                                                                                             
   • asym_beba_a1=0.0                                                                                                             
   • asym_beba_b0=0.0                                                                                                             
   • asym_beba_b1=0.0                                                                                                             


Peak profile type for experiment 'y2o3' changed to


pseudo-voigt + berar-baldinozzi asymmetry


Absorption type changed to


cylinder-hewat


## edi-cryspy VS FullProf

In [7]:
experiment.calculator.type = 'cryspy'

project.analysis.calculate()
calc_ed_cryspy = experiment.data.intensity_calc
LABEL_ED_CRYSPY = verify.engine_label('cryspy')

project.display.pattern_comparison(
    'y2o3',
    reference=verify.restrict_to_included(experiment, calc_fullprof),
    candidate=calc_ed_cryspy,
    reference_label=FULLPROF_LABEL,
    candidate_label=LABEL_ED_CRYSPY,
)

Calculator for experiment 'y2o3' already set to


cryspy


## Fit edi-cryspy to FullProf

In [8]:
experiment.peak.asym_beba_a0.free = True
experiment.peak.asym_beba_b0.free = True

project.analysis.fit()
project.display.fit.results()

project.analysis.calculate()
calc_ed_cryspy_refined = experiment.data.intensity_calc
LABEL_ED_CRYSPY_REFINED = verify.engine_label('cryspy', note='refined')

project.display.pattern_comparison(
    'y2o3',
    reference=verify.restrict_to_included(experiment, calc_fullprof),
    candidate=calc_ed_cryspy_refined,
    reference_label=FULLPROF_LABEL,
    candidate_label=LABEL_ED_CRYSPY_REFINED,
)

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'y2o3' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


...

,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,lmfit (leastsq)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),2.19
4,🔁 Iterations,7
5,📏 Goodness-of-fit (reduced χ²),6.63
6,"📏 R-factor (Rf, %)",0.22
7,"📏 R-factor squared (Rf², %)",0.19
8,"📏 Weighted R-factor (wR, %)",0.19


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,y2o3,peak,,asym_beba_a0,,0.1769,-0.2713,0.0003,253.30 % ↓
2,y2o3,peak,,asym_beba_b0,,0.0341,-0.0331,0.0001,197.04 % ↓


## Agreement check

In [9]:
verify.assert_patterns_agree(
    [
        (
            f'{LABEL_ED_CRYSPY_REFINED} vs {FULLPROF_LABEL}',
            verify.restrict_to_included(experiment, calc_fullprof),
            calc_ed_cryspy_refined,
        ),
    ],
)

,Comparison,Metric,Expected,Actual,OK
1,"edi 999.0.0 (cryspy 0.11.0, refined) vs FullProf 8.40",Profile diff (%),< 2.5,0.19,✅
2,,Max deviation (%),< 6,0.17,✅
3,,Area ratio,0.99 to 1.01,1.0007,✅
4,,Shape correlation,> 0.999,1.0000,✅


True